In [1]:
import argparse
from dataclasses import dataclass
from pathlib import Path

from omegaconf import OmegaConf
import torch

from src.lightning.lit_gpt import LitGPT
from src.models.lm import DecoderLM
from src.tokenizers.BPE import BytePairEncoding
from src.utils import paths

#OmegaConf.register_resolver("paths", lambda name: str(getattr(paths, name)))
paths.EXPERIMENTS_DIR = Path("/home/mcvjetko/phd/projects/transformer/experiments")

@dataclass
class GenConfig:
    # train_config: str = str(paths.EXPERIMENTS_DIR / "1B-model/test-2-20261021-proper-velocity/config.resolved.yaml")
    # ckpt: str = str(paths.EXPERIMENTS_DIR / "1B-model/test-2-20261021-proper-velocity/last.ckpt")
    train_config: str = str(paths.EXPERIMENTS_DIR / "1B-model/as-240B-20261025-breezy-sound/config.resolved.yaml")
    ckpt: str = str(paths.EXPERIMENTS_DIR / "1B-model/as-240B-20261025-breezy-sound/last.ckpt")
    tokenizer: str = str(paths.EXPERIMENTS_DIR / "1B-model/hr_en_bs_tokenizer_48000")
    prompt: str = "Test Prompt"
    max_new_tokens: int = 64
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def load_config(argv: list[str] | None = None):

    # Register the "paths" resolver for path interpolation if needed
    cli = OmegaConf.from_cli(argv)
    cfg = OmegaConf.structured(GenConfig)
    cfg = OmegaConf.merge(cfg, cli)  # external file overrides
    # Always load train_config from (potentially overridden) cfg.train_config
    cfg_train = OmegaConf.load(cfg.train_config)
    return OmegaConf.to_object(cfg), cfg_train


/home/mcvjetko/phd/projects/transformer/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cfg, cfg_train = load_config([])
tokenizer = BytePairEncoding.from_file(cfg.tokenizer)
# Get <HR> and <EN> token ids from tokenizer special vocab. Fallback to 3 if not found.
translate_token_id = tokenizer._special_vocab.get("<TRANSLATE>", 3)
hr_token_id = tokenizer._special_vocab.get("<HR>", 3)
en_token_id = tokenizer._special_vocab.get("<EN>", 3)
prompt_ids = tokenizer.tokenize(cfg.prompt, add_special=False)
# Only wrap prompt_ids with language tokens if both <HR> and <EN> are found, and prompt_ids is nonempty
if not isinstance(prompt_ids, list) or len(prompt_ids) == 0:
    raise SystemExit("tokenizer produced an empty prompt")
#prompt_ids = [hr_token_id] + prompt_ids + [en_token_id]
#prompt_ids = [translate_token_id] + [en_token_id] + prompt_ids + [hr_token_id]
prompt_ids = [hr_token_id] + prompt_ids
print(cfg.prompt)

model = DecoderLM(**cfg_train.model)
lit = LitGPT.load_from_checkpoint(
    transformer=model,
    checkpoint_path=cfg.ckpt,
    map_location="cpu",
    weights_only=False,
    strict=False,
)
model = lit.transformer.to(cfg.device).eval()
del lit

tokens = torch.tensor([prompt_ids], dtype=torch.long, device=cfg.device)
print(tokens.shape)
with torch.no_grad():
    tokens = model.generate(
        tokens,
        temperature=0.5,
        seq_len=96,
        top_k=32,
        top_p=0.9,
    )

full = tokenizer.decode(tokens[0].tolist(), add_special=True)
# Find the index of the first occurrence of <EOS>
eos_index = full.find("<EOS>")

print("\n")
print("\n")
print("\n")
if eos_index != -1:
    full = full[:eos_index]
print(full)
# print("--- continuation ---")
# print(continuation)



Test Prompt
lightning version: 2.6.6


/home/mcvjetko/phd/projects/transformer/.venv/lib/python3.11/site-packages/lightning/pytorch/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['bits_per_byte.id_to_len']


torch.Size([1, 4])






<HR>Test Prompt: "Tražim nešto"
U ovom testu Prompt, na slici ispod, možete vidjeti kako se ponaša uređaj Galaxy A5 (2016) kada je ekran zaključan. Ukoliko je ekran zaključan, uređaj će prikazati ovu poruku.
Kako bi riješili ovaj test, potrebno je da uđete u Postavke, zatim na opciju Sigurnost, a zatim i na opciju Zaključaj ekran. Nakon toga, uđite u Postavke i


In [12]:
prompt = ("2+9=")

prompt_ids = tokenizer.tokenize(prompt, add_special=False)
#prompt_ids = [hr_token_id] + prompt_ids + [en_token_id]
#prompt_ids = [translate_token_id] + [en_token_id] + prompt_ids + [hr_token_id]
#prompt_ids = [hr_token_id] + prompt_ids
prompt_ids = [en_token_id] + prompt_ids


tokens = torch.tensor([prompt_ids], dtype=torch.long, device=cfg.device)
print(tokens.shape)
with torch.no_grad():
    tokens = model.generate(
        tokens,
        temperature=0.1,
        seq_len=96,
        top_k=32,
        top_p=0.9,
    )

full = tokenizer.decode(tokens[0].tolist(), add_special=True)
# Find the index of the first occurrence of <EOS>
eos_index = full.find("<EOS>")
if eos_index != -1:
    full = full[:eos_index]

import textwrap
wrapped = textwrap.fill(full, width=100)  # wrap lines at 100 characters
print(wrapped)

torch.Size([1, 5])
<EN>2+9=10 10+9=10 10+10=10 10+10=10 10+10=10 10+10=10 10+10=10 10+10=10 10+10=10 10+10=10 10+10=10
